# Session 11 · Guided Lab — Evaluation and Continuous Model Improvement
*Desarrollo y Evaluación de Modelos Propios · Especialización en IA Generativa y Desarrollo de Negocios · Universidad del Rosario*

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/11-continuous-improvement/lab.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/11-continuous-improvement/lab.ipynb)

**Scenario.** *Andes Bank* (a fictional digital bank) routes customer messages to 12 support queues with an intent classifier. A labelling vendor delivered a new training set, and roughly **1 in 10 labels is wrong**. Your job today is to run one full turn of the **improvement loop**: measure → analyse errors → form hypotheses → apply targeted fixes → re-measure → decide, and to leave a clean **experiment log** behind.

### Learning objectives
1. Build a baseline and an **experiment log** with a decision rule written *before* the experiments (RAE 2).
2. Run **error analysis by slice** (intent, text length, confused pairs) and turn it into hypotheses (RAE 2).
3. Find likely **mislabelled examples** with cross-validated predicted probabilities, an idea inspired by confident learning (RAE 2, RAE 6).
4. Tune hyperparameters with **Optuna** without touching the test set (RAE 2).
5. Write **CheckList-style behavioural tests** and report pass rates (RAE 2, RAE 6).
6. Measure what **int8 dynamic quantization** changes in size, latency and accuracy (RAE 2, RAE 6).

### Time plan (60 min)
| Part | Topic | Minutes |
|---|---|---|
| 1 | Baseline with noisy labels and the experiment log | 8 |
| 2 | Error analysis by slice | 9 |
| 3 | Data-centric fixes: suspicious labels and augmentation | 12 |
| 4 | Hyperparameter search with Optuna | 8 |
| 5 | Behavioural tests (CheckList-style) | 8 |
| 6 | Efficiency: int8 dynamic quantization | 9 |
| 7 | Final report on the test set | 6 |

### ✅ Kaggle checklist
- **Settings → Internet: on** (the dataset and the DistilBERT model are downloaded from the Hugging Face Hub).
- **Accelerator: none.** Everything today runs on CPU, and quantization is a CPU technique. Save your GPU quota.
- Run the cells in order with **Shift + Enter**. ✋ **Checkpoints** are answered in the Zoom chat; 🧪 **Try it** exercises are stretch goals with a hidden solution.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("optuna")
ensure("datasets")

## Part 1 — Baseline with noisy labels and the experiment log (8')

We use a 12-intent subset of **Banking77** (Casanueva et al., 2020), a public dataset of short customer-support messages to a bank. The subset mixes intents that are easy to separate (`exchange_rate`, `top_up_failed`) with pairs that even humans confuse (`card_arrival` vs `card_delivery_estimate`, `lost_or_stolen_card` vs `compromised_card`).

**Three splits, three jobs** (slides, Part 2):
- **train**: the model learns from it. Here it carries the vendor's label errors.
- **validation**: every decision today (keep or discard a change, pick hyperparameters) is made on it.
- **test**: touched **once**, at the very end, to report all versions. Never used to decide anything.

> **Simulation note.** We *inject* the label errors ourselves, so we know which labels are wrong. That lets us measure how well our detector works, something you can never do with real data. Validation and test keep their original labels: in practice, invest in a small, carefully reviewed **gold set** for evaluation.

In [ ]:
import time, re, string, tempfile, warnings, statistics, collections
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# Sizes: the smoke test (FAST_DEV_RUN) runs the same code on smaller numbers.
MAX_PER_INTENT = 60 if FAST_DEV_RUN else None     # training examples kept per intent (None = all)
MAX_TEST_PER_INTENT = 20 if FAST_DEV_RUN else None
NOISE_RATE = 0.10                                  # share of training labels the "vendor" got wrong
N_OPTUNA_TRIALS = 3 if FAST_DEV_RUN else 20
N_BOOTSTRAP = 200 if FAST_DEV_RUN else 1000
N_SST2 = 64 if FAST_DEV_RUN else 400               # sentiment examples for the quantization check
N_LATENCY_RUNS = 5 if FAST_DEV_RUN else 30

INTENTS = [
    "activate_my_card", "card_arrival", "card_delivery_estimate", "lost_or_stolen_card",
    "compromised_card", "declined_card_payment", "card_payment_not_recognised", "pending_transfer",
    "transfer_not_received_by_recipient", "cancel_transfer", "exchange_rate", "top_up_failed",
]
pd.set_option("display.max_colwidth", 90)

In [ ]:
banking = load_dataset("legacy-datasets/banking77")
LABEL_NAMES = banking["train"].features["label"].names

def to_frame(split, max_per_intent=None):
    """Keep our 12 intents, as a DataFrame with columns text and intent."""
    df = split.to_pandas()
    df["intent"] = [LABEL_NAMES[i] for i in df["label"]]
    df = df[df["intent"].isin(INTENTS)][["text", "intent"]]
    if max_per_intent:
        df = df.sample(frac=1, random_state=SEED).groupby("intent").head(max_per_intent)
    return df.reset_index(drop=True)

pool_df = to_frame(banking["train"], MAX_PER_INTENT)
test_df = to_frame(banking["test"], MAX_TEST_PER_INTENT)
train_df, valid_df = train_test_split(pool_df, test_size=0.2, stratify=pool_df["intent"], random_state=SEED)
train_df, valid_df = train_df.reset_index(drop=True), valid_df.reset_index(drop=True)
print(f"train {len(train_df)} · validation {len(valid_df)} · test {len(test_df)}")
train_df["intent"].value_counts().rename("training examples").to_frame().T

Now the "vendor" makes mistakes: we replace 10% of the training labels with a different intent chosen at random. Real label noise is rarely this uniform (annotators confuse *similar* intents more often), which is why confident learning models a full noise matrix. Uniform noise keeps today's simulation simple.

In [ ]:
def inject_label_noise(labels, rate, classes, seed):
    """Return (noisy_labels, flipped_mask): `rate` of the labels replaced by another class."""
    rng_noise = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=object)
    flip = rng_noise.choice(len(labels), size=int(round(rate * len(labels))), replace=False)
    noisy = labels.copy()
    for i in flip:
        noisy[i] = rng_noise.choice([c for c in classes if c != labels[i]])
    return noisy, noisy != labels

train_df["label"], train_df["is_flipped"] = inject_label_noise(train_df["intent"], NOISE_RATE, INTENTS, SEED)
print(f"Wrong training labels: {train_df['is_flipped'].sum()} of {len(train_df)}")
train_df.loc[train_df["is_flipped"], ["text", "intent", "label"]].head(5).rename(
    columns={"intent": "true intent (hidden in real life)", "label": "label we received"})

From now on the model only sees `label` (the noisy column). `intent` is the hidden truth, used only to *audit* our detector.

**The experiment log.** Every change is one row: hypothesis, change, validation metrics, the difference with the current champion and a decision. The **decision rule is fixed before running anything**: keep a change only if validation macro-F1 improves by at least `MIN_GAIN`. With ~300 validation messages, one message is worth about 0.3 points of accuracy, so smaller differences are mostly noise.

In [ ]:
def make_model(C=10.0, ngram_max=1, analyzer="word"):
    """TF-IDF + logistic regression. analyzer='word' uses n-grams (1, ngram_max);
    analyzer='char_wb' uses character n-grams (2, ngram_max) inside word boundaries."""
    low = 1 if analyzer == "word" else 2
    return make_pipeline(
        TfidfVectorizer(analyzer=analyzer, ngram_range=(low, ngram_max), sublinear_tf=True),
        LogisticRegression(C=C, max_iter=3000))

def evaluate(model, df):
    pred = model.predict(df["text"])
    return {"macro_f1": f1_score(df["intent"], pred, average="macro"), "accuracy": accuracy_score(df["intent"], pred)}

MIN_GAIN = 0.01   # decision rule, written down BEFORE the experiments
LOG, MODELS, TRAIN_SETS = [], {}, {}

def champion():
    """Id of the current best version: the last one that was kept."""
    return [r["id"] for r in LOG if r["decision"] in ("baseline", "keep")][-1]

def log_experiment(exp_id, hypothesis, change, model, train_texts, train_labels, notes=""):
    """Evaluate on validation, compare with the champion, apply the decision rule and log."""
    m = evaluate(model, valid_df)
    if LOG:
        best = next(r for r in LOG if r["id"] == champion())["val_macro_f1"]
        delta = m["macro_f1"] - best
        decision = "keep" if delta >= MIN_GAIN else "discard"
    else:
        delta, decision = 0.0, "baseline"
    LOG.append({"id": exp_id, "hypothesis": hypothesis, "change": change,
                "val_macro_f1": round(m["macro_f1"], 4), "val_accuracy": round(m["accuracy"], 4),
                "delta_vs_champion": round(delta, 4), "decision": decision, "notes": notes})
    MODELS[exp_id], TRAIN_SETS[exp_id] = model, (np.asarray(train_texts, dtype=object), np.asarray(train_labels, dtype=object))
    return pd.DataFrame(LOG)

**Baseline.** The team reuses the configuration of last quarter's pilot, which was labelled in-house: unigram TF-IDF and a weakly regularised logistic regression (`C=10`).

In [ ]:
baseline = make_model().fit(train_df["text"], train_df["label"])
log_experiment("E0", "—", "Baseline: word unigrams, C=10, vendor labels", baseline,
               train_df["text"], train_df["label"], notes="pilot configuration")

For teaching only, here is the same model trained on the *true* labels. This "oracle" is impossible in real life, but it tells us how much the label errors cost.

In [ ]:
oracle = make_model().fit(train_df["text"], train_df["intent"])
print(f"Validation macro-F1 · noisy labels: {evaluate(baseline, valid_df)['macro_f1']:.4f} "
      f"· true labels (oracle): {evaluate(oracle, valid_df)['macro_f1']:.4f}")

✋ **Checkpoint (chat).** How many macro-F1 points do the vendor's errors cost? Is that gap big enough to justify a week of relabelling? What else would you need to know to answer (hint: volume of messages, cost of a misrouted message)?

## Part 2 — Error analysis by slice (9')

A single average hides where the model fails. We slice the **validation** errors three ways: by intent, by confused pair and by message length. Each slice should end in a *hypothesis* you can test.

In [ ]:
val_pred = baseline.predict(valid_df["text"])
per_intent = pd.DataFrame({
    "f1": f1_score(valid_df["intent"], val_pred, average=None, labels=INTENTS),
    "support": valid_df["intent"].value_counts().reindex(INTENTS).to_numpy(),
    "train_examples": train_df["label"].value_counts().reindex(INTENTS).to_numpy(),
}, index=INTENTS).sort_values("f1")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(per_intent.index, per_intent["f1"], color="#0F766E")
ax.set_xlabel("F1 on validation"); ax.set_xlim(0, 1); ax.set_title("Baseline: F1 per intent")
plt.tight_layout(); plt.show()
per_intent.round(3)

In [ ]:
cm = confusion_matrix(valid_df["intent"], val_pred, labels=INTENTS)
pairs = [(INTENTS[i], INTENTS[j], cm[i, j]) for i in range(len(INTENTS)) for j in range(len(INTENTS)) if i != j and cm[i, j] > 0]
pd.DataFrame(pairs, columns=["true intent", "predicted as", "count"]).sort_values("count", ascending=False).head(8)

In [ ]:
def length_bucket(text):
    n = len(text.split())
    return "short (≤ 6 words)" if n <= 6 else ("medium (7–14)" if n <= 14 else "long (≥ 15)")

slices = valid_df.assign(pred=val_pred, bucket=valid_df["text"].map(length_bucket))
slices["correct"] = slices["pred"] == slices["intent"]
slices.groupby("bucket").agg(messages=("correct", "size"), accuracy=("correct", "mean")).round(3)

In [ ]:
# Read the errors themselves: numbers tell you where, examples tell you why.
slices.loc[~slices["correct"], ["text", "intent", "pred"]].head(10)

✋ **Checkpoint (chat).** Write one hypothesis in the form *"errors in slice X happen because Y, so change Z should help"*. For example: "`card_arrival` is confused with `card_delivery_estimate` because both mention *card* and *when*, so more distinguishing examples should help". Which of your hypotheses points to the **data**, and which to the **model**?

🧪 **Try it.** Add a fourth slice: messages that contain a question mark vs those that don't. Is accuracy different?

<details><summary>Show a solution</summary>

```python
slices["has_question"] = slices["text"].str.contains(r"\?")
slices.groupby("has_question").agg(messages=("correct", "size"), accuracy=("correct", "mean")).round(3)
```
A slice only matters if it is large enough and its gap is bigger than the noise: with a few dozen messages, a 3-point difference may be chance.
</details>

## Part 3 — Data-centric fixes (12')

### 3a · Find likely label errors with out-of-sample probabilities

Idea (inspired by **confident learning**, Northcutt, Jiang & Chuang, 2021): train on part of the data, predict the rest, and look at how much probability the model gives to the label we *received*. A model never sees the example it is scoring, so it cannot simply memorise a wrong label.

1. `cross_val_predict(..., method="predict_proba")` gives an **out-of-sample** probability vector $\hat{p}(x_i)$ for every training example (5 folds).
2. **Self-confidence** of example $i$ with received label $\tilde{y}_i$: $\;s_i = \hat{p}(\tilde{y}_i \mid x_i)$.
3. **Per-class threshold** $t_j$ = average self-confidence of the examples labelled $j$ (some intents are simply harder).
4. Example $i$ is **flagged** when another class $j \ne \tilde{y}_i$ is the most probable class among those with $\hat{p}_j(x_i) \ge t_j$. This is a simplified version of the paper's *confident joint*; for real projects use the authors' open-source library (`cleanlab`), which adds calibration and ranking options we skip here.

In [ ]:
labels_rx = train_df["label"].to_numpy(dtype=object)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
proba = cross_val_predict(make_model(), train_df["text"], labels_rx, cv=cv, method="predict_proba")
classes = np.unique(labels_rx)                       # column order used by predict_proba
given = np.searchsorted(classes, labels_rx.astype(str))
self_conf = proba[np.arange(len(proba)), given]
thresholds = np.array([self_conf[given == j].mean() for j in range(len(classes))])

above = proba >= thresholds                          # classes each example "confidently" belongs to
suggested = np.where(above, proba, -1).argmax(axis=1)
flagged = above.any(axis=1) & (suggested != given)

train_df["self_conf"] = self_conf.round(3)
train_df["suggested"] = classes[suggested]
train_df["flagged"] = flagged
print(f"Flagged {flagged.sum()} of {len(train_df)} training examples as likely label errors.")
train_df.loc[flagged].sort_values("self_conf")[["text", "label", "suggested", "self_conf"]].head(10)

Because this is a simulation, we can audit the detector against the hidden truth. With real data you would **review a sample by hand** instead.

In [ ]:
tp = (train_df["flagged"] & train_df["is_flipped"]).sum()
precision = tp / max(train_df["flagged"].sum(), 1)
recall = tp / max(train_df["is_flipped"].sum(), 1)
print(f"Detector precision {precision:.2f} (flagged labels that really were wrong)")
print(f"Detector recall    {recall:.2f} (wrong labels that were flagged)")

Two fixes, two log rows. **Drop** removes the flagged examples; **relabel** replaces their label with the suggested one. Relabelling keeps more data but trusts the model's suggestion; in production a human confirms each change.

In [ ]:
keep = ~train_df["flagged"]
model_drop = make_model().fit(train_df.loc[keep, "text"], train_df.loc[keep, "label"])
log_experiment("E1", "About 10% of vendor labels are wrong; flagged examples pull the decision boundary",
               "Drop flagged examples (CL-inspired, 5-fold)", model_drop,
               train_df.loc[keep, "text"], train_df.loc[keep, "label"], notes=f"{flagged.sum()} dropped")

relabelled = np.where(train_df["flagged"], train_df["suggested"], train_df["label"])
model_relabel = make_model().fit(train_df["text"], relabelled)
log_experiment("E2", "Same as E1, but keeping the data is worth more than removing it",
               "Relabel flagged examples with the suggested intent", model_relabel,
               train_df["text"], relabelled, notes="no human review in this simulation")

### 3b · Augment the weakest intents

A second, common data-centric move: create extra training examples for the intents with the lowest validation F1. We use four cheap text edits in the spirit of *EDA* (Wei & Zou, 2019): synonym replacement, swapping two neighbouring words, deleting a word, and adding a greeting. We apply them to the **current champion's** training data.

In [ ]:
SYNONYMS = {"card": ["debit card"], "money": ["funds", "cash"], "transfer": ["payment"], "want": ["would like"],
            "need": ["want"], "help": ["assist"], "why": ["how come"], "still": ["yet"], "received": ["got"]}
GREETINGS = ["Hi,", "Hello,", "Please help:", "Quick question:"]

def augment(text, rnd):
    """One random EDA-style edit of a message (label-preserving by assumption)."""
    words = text.split()
    op = rnd.choice(["synonym", "swap", "delete", "greeting"])
    if op == "synonym":
        spots = [i for i, w in enumerate(words) if w.lower().strip("?.!,") in SYNONYMS]
        if spots:
            i = rnd.choice(spots)
            words[i] = rnd.choice(SYNONYMS[words[i].lower().strip("?.!,")])
    elif op == "swap" and len(words) > 3:
        i = rnd.randrange(len(words) - 1)
        words[i], words[i + 1] = words[i + 1], words[i]
    elif op == "delete" and len(words) > 4:
        del words[rnd.randrange(len(words))]
    else:
        words = [rnd.choice(GREETINGS)] + words
    return " ".join(words)

champ_texts, champ_labels = TRAIN_SETS[champion()]
champ_f1 = f1_score(valid_df["intent"], MODELS[champion()].predict(valid_df["text"]), average=None, labels=INTENTS)
weak = [INTENTS[i] for i in np.argsort(champ_f1)[:3]]
rnd = random.Random(SEED)
new_pairs = [(augment(t, rnd), y) for t, y in zip(champ_texts, champ_labels) if y in weak]
aug_texts = np.concatenate([champ_texts, [t for t, _ in new_pairs]])
aug_labels = np.concatenate([champ_labels, [y for _, y in new_pairs]])
print(f"Champion: {champion()} · weakest intents: {weak} · {len(new_pairs)} new examples")
for t, y in new_pairs[:3]:
    print(f"  [{y}] {t}")

model_aug = make_model().fit(aug_texts, aug_labels)
log_experiment("E3", f"The weakest intents ({', '.join(weak)}) need more examples",
               "EDA-style augmentation ×1 of the 3 weakest intents", model_aug, aug_texts, aug_labels)

✋ **Checkpoint (chat).** Did augmentation clear the `MIN_GAIN` bar? Look back at the confused pairs in Part 2: do paraphrases of existing messages add the information the model is missing to separate, say, `card_arrival` from `card_delivery_estimate`?

🧪 **Try it.** Write five *new* messages for the weakest intent that contain the words that distinguish it from its confused partner, add them to the champion's data and retrain. (This is **targeted data collection**, the data-centric move that usually pays.)

<details><summary>Show a solution</summary>

```python
extra = ["My card still hasn't arrived and it's been two weeks",
         "The card you posted never got to my house",
         "Card not delivered yet, is it lost in the mail?",
         "I ordered a card last month and nothing has arrived",
         "Where is the card you sent me? It never came"]
t, y = TRAIN_SETS[champion()]
m = make_model().fit(np.concatenate([t, extra]), np.concatenate([y, ["card_arrival"] * len(extra)]))
log_experiment("E3b", "card_arrival lacks 'not arrived' phrasing", "5 hand-written card_arrival examples", m,
               np.concatenate([t, extra]), np.concatenate([y, ["card_arrival"] * len(extra)]))
```
Adjust the intent name and the messages to *your* weakest intent. Five messages rarely move macro-F1 by a full point; the lesson is the method.
</details>

## Part 4 — Hyperparameter search with Optuna (8')

**Optuna** (Akiba et al., 2019) builds the search space *while* the objective runs ("define-by-run"), so the space can branch: word n-grams use one range, character n-grams another. Its default sampler, **TPE**, proposes new trials near the ones that scored well; random search (Bergstra & Bengio, 2012) is the baseline it should beat.

The objective trains on the champion's training data and returns **validation** macro-F1. The test set is not in the objective, and never will be.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
opt_texts, opt_labels = TRAIN_SETS[champion()]

def objective(trial):
    analyzer = trial.suggest_categorical("analyzer", ["word", "char_wb"])
    if analyzer == "word":
        ngram_max = trial.suggest_int("word_ngram_max", 1, 3)
    else:
        ngram_max = trial.suggest_int("char_ngram_max", 3, 5)
    C = trial.suggest_float("C", 0.1, 100, log=True)
    model = make_model(C=C, ngram_max=ngram_max, analyzer=analyzer).fit(opt_texts, opt_labels)
    return evaluate(model, valid_df)["macro_f1"]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
t0 = time.time()
study.optimize(objective, n_trials=N_OPTUNA_TRIALS)
print(f"{N_OPTUNA_TRIALS} trials in {time.time() - t0:.1f}s · best validation macro-F1 {study.best_value:.4f}")
print("Best parameters:", study.best_params)

In [ ]:
values = [t.value for t in study.trials]
plt.figure(figsize=(7, 3.5))
plt.plot(range(1, len(values) + 1), values, "o", color="#0F766E", label="trial")
plt.plot(range(1, len(values) + 1), np.maximum.accumulate(values), "-", color="#9E1B32", label="best so far")
plt.xlabel("trial"); plt.ylabel("validation macro-F1"); plt.title("Optuna optimisation history"); plt.legend()
plt.tight_layout(); plt.show()

bp = study.best_params
best_ngram = bp.get("word_ngram_max", bp.get("char_ngram_max"))
model_opt = make_model(C=bp["C"], ngram_max=best_ngram, analyzer=bp["analyzer"]).fit(opt_texts, opt_labels)
log_experiment("E4", "The pilot's C=10 and unigrams are not optimal for this data",
               f"Optuna TPE, {N_OPTUNA_TRIALS} trials: {bp['analyzer']}, n-grams up to {best_ngram}, C={bp['C']:.2f}",
               model_opt, opt_texts, opt_labels, notes="selected on validation: expect optimism")

✋ **Checkpoint (chat).** The best of 20 trials was *selected* on the validation set. Why is its validation score probably a little optimistic? (Think of picking the best of 20 students' guesses and calling it the class average.) Where will we see the honest number?

🧪 **Try it.** Replace `TPESampler` with `optuna.samplers.RandomSampler(seed=SEED)` and rerun the search. Does TPE beat random search with the same budget?

<details><summary>Show a solution</summary>

```python
random_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler(seed=SEED))
random_study.optimize(objective, n_trials=N_OPTUNA_TRIALS)
print(f"TPE {study.best_value:.4f} vs random {random_study.best_value:.4f}")
```
With a small space and only 20 trials the two are often close; TPE's advantage grows with bigger spaces and budgets. Compare several seeds before concluding anything.
</details>

## Part 5 — Behavioural tests, CheckList-style (8')

Accuracy tells you *how often* a model is right on data like the test set. **Behavioural tests** (Ribeiro et al., 2020) check *capabilities* with targeted inputs:
- **INV (invariance):** a label-preserving change (typo, casing, irrelevant sentence) should **not** change the prediction.
- **MFT (minimum functionality):** simple cases with a known answer, like a unit test.
- **DIR (directional):** a change should move the prediction in a known direction.

The pass rate of an INV test is the share of messages whose prediction stays the same after the perturbation, whether or not the original prediction was right.

In [ ]:
IRRELEVANT = ["By the way, I love your app.", "I'm writing from my phone.", "Thanks in advance!",
              "It is raining here today.", "Sorry for my English."]

def add_typo(text, rnd):
    """Swap two neighbouring letters inside one word of 4+ letters."""
    words = text.split()
    spots = [i for i, w in enumerate(words) if len(w) >= 4]
    if not spots:
        return text
    i = rnd.choice(spots)
    w = words[i]
    j = rnd.randrange(len(w) - 1)
    words[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    return " ".join(words)

def add_irrelevant(text, rnd):
    return f"{text.rstrip()} {rnd.choice(IRRELEVANT)}"

def no_punctuation(text, rnd):
    return text.translate(str.maketrans("", "", string.punctuation))

PERTURBATIONS = {"typo": add_typo, "UPPERCASE": lambda t, rnd: t.upper(),
                 "irrelevant sentence": add_irrelevant, "no punctuation": no_punctuation}

def invariance_report(predict, texts, seed=SEED):
    """Pass rate of every INV test, plus up to 3 failing examples per test."""
    original = np.asarray(predict(list(texts)))
    rows, failures = [], {}
    for name, perturb in PERTURBATIONS.items():
        rnd = random.Random(seed)
        changed = [perturb(t, rnd) for t in texts]
        new = np.asarray(predict(changed))
        same = new == original
        rows.append({"test": f"INV · {name}", "pass_rate": round(float(same.mean()), 4), "n": len(texts)})
        failures[name] = [(changed[i], original[i], new[i]) for i in np.where(~same)[0][:3]]
    return pd.DataFrame(rows), failures

inv_texts = valid_df["text"].tolist()
report_base, _ = invariance_report(baseline.predict, inv_texts)
report_champ, fails_champ = invariance_report(MODELS[champion()].predict, inv_texts)
robustness = report_base.rename(columns={"pass_rate": "E0 baseline"}).merge(
    report_champ.rename(columns={"pass_rate": f"{champion()} champion"}), on=["test", "n"])
robustness

In [ ]:
for name, examples in fails_champ.items():
    for text, before, after in examples[:2]:
        print(f"[{name}] {text!r}: {before} → {after}")

Why does the UPPERCASE test pass for both models? Because `TfidfVectorizer` lowercases the text by default. A test that always passes is still useful: it documents a guarantee and will catch a regression if someone changes the preprocessing.

### 5b · Negation: an MFT for a sentiment model

Negation is **not** an invariance for intents ("I want to cancel my transfer" vs "I don't want to cancel it" may need different queues), but it is a classic capability test for **sentiment**. We load a ready-made sentiment classifier, `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (a DistilBERT fine-tuned on SST-2), which we will also quantize in Part 6.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
SENT_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
sent_tok = AutoTokenizer.from_pretrained(SENT_NAME)
sent_fp32 = AutoModelForSequenceClassification.from_pretrained(SENT_NAME).eval()   # stays on CPU
ID2LABEL = sent_fp32.config.id2label

def sentiment(model, texts, batch_size=32):
    """Labels (NEGATIVE/POSITIVE) for a list of texts, on CPU."""
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            enc = sent_tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=128, return_tensors="pt")
            out += [ID2LABEL[int(k)] for k in model(**enc).logits.argmax(-1)]
    return out

THINGS = ["the app", "the customer service", "the new card", "the transfer process", "the support team"]
POSITIVE_WORDS = ["good", "helpful", "fast", "easy to use", "reliable"]
NEGATED = [s for thing in THINGS for w in POSITIVE_WORDS for s in
           (f"{thing.capitalize()} is not {w}.", f"{thing.capitalize()} was not {w} at all.", f"I don't think {thing} is {w}.")]
PLAIN = [f"{thing.capitalize()} is {w}." for thing in THINGS for w in POSITIVE_WORDS]
NEGATIVE_WORDS = ["bad", "slow", "confusing", "useless", "terrible"]
NEGATED_NEG = [f"{thing.capitalize()} is not {w}." for thing in THINGS for w in NEGATIVE_WORDS]

def negation_tests(model):
    return {"MFT · plain positive → POSITIVE": np.mean([p == "POSITIVE" for p in sentiment(model, PLAIN)]),
            "MFT · negated positive → NEGATIVE": np.mean([p == "NEGATIVE" for p in sentiment(model, NEGATED)]),
            "MFT · negated negative → POSITIVE": np.mean([p == "POSITIVE" for p in sentiment(model, NEGATED_NEG)])}

neg_fp32 = negation_tests(sent_fp32)
{k: round(float(v), 4) for k, v in neg_fp32.items()}

In CheckList terms, "not bad" is harder than "not good": with only two labels, a negated negative has to go to POSITIVE, and models trained on reviews often key on the word *bad*. Look at which of the three MFTs is the weakest.

✋ **Checkpoint (chat).** Which INV test hurts the intent classifier most? Propose one change that could fix it, and say how you would check that the fix does not cost accuracy.

🧪 **Try it.** Character n-grams (`analyzer="char_wb"`) often make TF-IDF models more robust to typos. Train one on the champion's data and compare typo pass rates.

<details><summary>Show a solution</summary>

```python
t, y = TRAIN_SETS[champion()]
char_model = make_model(C=30, ngram_max=5, analyzer="char_wb").fit(t, y)
print(invariance_report(char_model.predict, inv_texts)[0])
print(evaluate(char_model, valid_df))
```
If Optuna already picked `char_wb`, compare with a word-level model instead. Robustness and accuracy can move in different directions: report both.
</details>

## Part 6 — Efficiency: int8 dynamic quantization (9')

**Dynamic quantization** stores the weights of every `nn.Linear` layer as 8-bit integers and quantizes activations on the fly at inference time (CPU only). We compare the original (float32) and quantized models on:
- **size on disk** (saved `state_dict`);
- **latency**: median of several runs for one message (batch 1) and for a batch of 32;
- **accuracy** on a sample of the SST-2 validation set, and the negation tests from Part 5.

PyTorch's eager-mode API (`torch.ao.quantization.quantize_dynamic`, also reachable as `torch.quantization.quantize_dynamic`) is marked as deprecated in recent releases in favour of the `torchao` library. It still works in the versions we tested; if your PyTorch no longer ships it, the next cell explains it and skips the comparison instead of crashing.

In [ ]:
warnings.filterwarnings("ignore", message=r".*quantiz")   # deprecation notices of the eager-mode API

def quantize_int8(model):
    """Dynamic int8 quantization of nn.Linear layers; returns None if this PyTorch cannot do it."""
    engines = torch.backends.quantized.supported_engines
    if torch.backends.quantized.engine not in engines or torch.backends.quantized.engine == "none":
        for engine in ("x86", "fbgemm", "qnnpack"):          # x86 servers first, ARM (e.g. Apple silicon) last
            if engine in engines:
                torch.backends.quantized.engine = engine
                break
    try:
        from torch.ao.quantization import quantize_dynamic
    except ImportError:
        try:
            from torch.quantization import quantize_dynamic
        except ImportError:
            print("This PyTorch has no eager-mode quantize_dynamic (migrated to torchao): skipping.")
            return None
    try:
        return quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)   # returns a quantized copy
    except Exception as exc:  # e.g. no quantized engine for this CPU
        print(f"Dynamic quantization is not available here ({type(exc).__name__}: {exc}): skipping.")
        return None

sent_int8 = quantize_int8(sent_fp32)
print("Quantized engine:", torch.backends.quantized.engine, "| quantized model ready:", sent_int8 is not None)

In [ ]:
def size_on_disk_mb(model):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "weights.pt")
        torch.save(model.state_dict(), path)
        return os.path.getsize(path) / 1e6

def median_latency_ms(model, texts, runs=N_LATENCY_RUNS):
    enc = sent_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    times = []
    with torch.inference_mode():
        model(**enc)                                   # warm-up run, not timed
        for _ in range(runs):
            start = time.perf_counter()
            model(**enc)
            times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times)

sst2 = load_dataset("stanfordnlp/sst2", split="validation").shuffle(seed=SEED).select(range(N_SST2))
sst2_texts, sst2_gold = list(sst2["sentence"]), [ID2LABEL[int(k)] for k in sst2["label"]]
one_message = ["The app is not easy to use and the support team was slow."]

rows, preds = [], {}
for name, model in [("float32", sent_fp32), ("int8 dynamic", sent_int8)]:
    if model is None:
        continue
    preds[name] = sentiment(model, sst2_texts)
    negation = negation_tests(model)
    rows.append({"version": name, "size_MB": round(size_on_disk_mb(model), 1),
                 "latency_ms_batch1": round(median_latency_ms(model, one_message), 2),
                 "latency_ms_batch32": round(median_latency_ms(model, sst2_texts[:32]), 1),
                 "sst2_accuracy": round(float(np.mean([p == g for p, g in zip(preds[name], sst2_gold)])), 4),
                 "negation_pass_rate": round(float(negation["MFT · negated positive → NEGATIVE"]), 4)})
efficiency = pd.DataFrame(rows)
if len(preds) == 2:
    agree = np.mean([a == b for a, b in zip(preds["float32"], preds["int8 dynamic"])])
    print(f"The two versions agree on {agree:.1%} of the {N_SST2} SST-2 sentences.")
efficiency

**How to read the table**
- **Size** shrinks, but by less than 4×: only the `nn.Linear` weights become int8; the embedding matrix and the layer norms stay in float32.
- **Latency** depends on the CPU and on PyTorch's int8 kernels. On many x86 servers int8 is faster; on some ARM laptops it can even be slower. Always measure on the hardware you will deploy on.
- **Batching** divides the cost per message: compare `latency_ms_batch32 / 32` with `latency_ms_batch1`. Batch jobs care about throughput; a live chat cares about the latency of one message.
- **Accuracy** on a few hundred sentences has a margin of error of a few points: a difference of one or two sentences is not a real loss.

✋ **Checkpoint (chat).** Andes Bank wants to run this sentiment model on a small CPU server with a 50 ms budget per message. Which numbers in the table would you show the product owner, and which extra measurement is missing?

## Part 7 — The experiment log and the final report (6')

1. The log records every decision with its evidence.
2. A **paired bootstrap** on the validation set tells us whether the champion's gain over the baseline is larger than the noise of a ~300-message evaluation.
3. Then, and only then, every version is scored **once** on the test set.

In [ ]:
log_df = pd.DataFrame(LOG)
log_df

In [ ]:
def paired_bootstrap_ci(y_true, pred_a, pred_b, n=N_BOOTSTRAP, seed=SEED):
    """95% interval for macro-F1(b) − macro-F1(a), resampling the same messages for both models."""
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    rng_boot = np.random.default_rng(seed)
    deltas = []
    for _ in range(n):
        idx = rng_boot.integers(0, len(y_true), len(y_true))
        deltas.append(f1_score(y_true[idx], pred_b[idx], average="macro") - f1_score(y_true[idx], pred_a[idx], average="macro"))
    return np.percentile(deltas, [2.5, 97.5])

champ_id = champion()
lo, hi = paired_bootstrap_ci(valid_df["intent"], baseline.predict(valid_df["text"]), MODELS[champ_id].predict(valid_df["text"]))
print(f"Champion {champ_id} vs E0 on validation: 95% interval for the macro-F1 gain [{lo:+.4f}, {hi:+.4f}]")
print("→ the gain is larger than the evaluation noise" if lo > 0 else "→ we cannot rule out that the gain is noise")

In [ ]:
# The test set, used once, to report every version (no decision is taken from this table).
final = log_df[["id", "change", "decision", "val_macro_f1"]].copy()
final["test_macro_f1"] = [round(evaluate(MODELS[i], test_df)["macro_f1"], 4) for i in final["id"]]
final["val_minus_test"] = (final["val_macro_f1"] - final["test_macro_f1"]).round(4)
out_dir = "/kaggle/working" if IN_KAGGLE else tempfile.gettempdir()
log_df.to_csv(os.path.join(out_dir, "s11_experiment_log.csv"), index=False)
print(f"Champion: {champ_id} · log saved to {out_dir}/s11_experiment_log.csv")
final

✋ **Checkpoint (chat).** Compare `val_macro_f1` and `test_macro_f1` for the champion and for the Optuna row. Which gain survived on test, and which one shrank? What does that say about selecting the best of many trials on one validation set?

## Wrap-up

- **Loop, not line:** baseline → error analysis → hypothesis → targeted fix → re-measure → decide with a rule written in advance.
- **Data first:** with noisy labels, finding and fixing wrong labels usually beats tuning; paraphrasing existing examples rarely adds the information a confused pair needs.
- **Honest numbers:** decide on validation, report on test once, and check whether a gain is larger than the evaluation noise.
- **Beyond accuracy:** behavioural tests (INV, MFT, DIR) and efficiency (size, latency, batching) are part of "better".

**Next: the Session Challenge.** You will run the same loop on *your* intents, noise rate and trial budget, write a robustness report and decide which version to deploy under *your* latency, accuracy and size constraints.

**Project milestone H4 (groups, already submitted):** your improvement iteration went in on Friday 23 Oct at 23:59, in the experiment log of the technical report (§5) that is part of the final deliverable. Nothing project-related is due today: use today's checklist (one change, rule written in advance, decided on validation, gain above the noise) to review it with your group and plan what to say about it in the S12 pitch, which starts at 10:00 today. See `proyecto-final/README.md` in the course repository.

**References:** Casanueva et al. (2020) · Northcutt, Jiang & Chuang (2021) · Wei & Zou (2019) · Akiba et al. (2019) · Bergstra & Bengio (2012) · Ribeiro et al. (2020) · Jacob et al. (2018) · Sanh et al. (2019). Full list in the slides.